# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. All entities such as record sets, fields, and columns are referenced using their Croissant schema `@id` fields for consistency.

### Dataset Source

Croissant schema URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset high-level description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields. All references are by `@id`.

We'll enumerate the available record sets, their field ids, and a preview of sample records.

In [ ]:
# List all record sets, their fields by @id
record_sets = dataset.metadata.record_sets

print("Found record sets:")
for rset in record_sets:
    print(f"- Record set @id: {rset.id}, name: {rset.name if hasattr(rset, 'name') else ''}")
    if hasattr(rset, 'fields'):
        print("  Fields (@id):", [field.id for field in rset.fields])

# Preview first 1-2 records for each record set
for rset in record_sets:
    print(f"\nSample records from record set '{rset.id}':")
    for i, rec in enumerate(dataset.records(record_set=rset.id)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load all tabular data from the available record sets into DataFrames for further analysis. We use the `@id` of each record set for extraction, matching the entities found above.

In [ ]:
# Extract data from each record set into a DataFrame, referencing entities by @id
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# List of record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        dataframes[rsid] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set {rsid} with shape {dataframes[rsid].shape}")
    else:
        print(f"No records found for record set {rsid}")

# For demonstration, use the first record set with data
usable_rsids = [k for k in dataframes]
if usable_rsids:
    main_record_set_id = usable_rsids[0]
    print(f"Using main record set: {main_record_set_id}")
    print("Columns (by field @id):", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Process the data: filter records, normalize a numeric field, and group by a categorical field.

We demonstrate by picking a numeric field (by `@id`) and a group/categorical field (by `@id`).

In [ ]:
import numpy as np

# Use fields from the main DataFrame, referenced by their @id (column names)
df = dataframes[main_record_set_id].copy()

# Heuristically pick a numeric field @id
numeric_field_id = None
for col in df.columns:
    # Try to detect typical numeric field
    if df[col].dtype in [np.float32, np.float64, np.int32, np.int64]:
        numeric_field_id = col
        break
    # Try to convert
    try:
        df[col] = pd.to_numeric(df[col])
        if df[col].notnull().sum() > 0:
            numeric_field_id = col
            break
    except Exception:
        continue

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean()  # Use mean as a threshold for demonstration
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records: {len(filtered_df)} rows with field @{numeric_field_id} above its mean ({threshold:.2f})")

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to pick a group field
    group_field_id = None
    for col in df.columns:
        if col == numeric_field_id:
            continue
        if df[col].dtype == object and df[col].nunique() < 20:
            group_field_id = col
            break

    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("Could not find a suitable numeric field for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationships in the grouped data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Distribution plot
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of @{numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Count")
    plt.show()

    # If group plot is available
    if group_field_id is not None:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Average of @{numeric_field_id} by @{group_field_id}")
        plt.xlabel(f"{group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No numeric field available to visualize.")

## 6. Conclusion
We have demonstrated how to load, explore, and analyze the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using `mlcroissant`.

Key steps included referencing all record sets, fields, and columns via their `@id`; extracting records into DataFrames; basic data cleaning and normalization; and visualizing distribution and group-level statistics. For robust analysis, consult the full Croissant schema and FAIR² metadata to select appropriate fields for downstream tasks.